## Spot rate estimation 

### 1. Bootstraping

Decomposition of swap contract into two bond

In [17]:
# Using Hypothetical data 
# Assuming 3M spot rate = 0
# I guess the complication about 3M spot rate is coming from the payment schedule

import pandas as pd 
import numpy as np 
import sympy as sp

df_swap_market = pd.DataFrame(
    columns = ["Maturity", "Frequency", "Swap rate", "Spot rate", "Forward rate", "Spot rate soln", "Discount soln"],
    data = [
        [0.25, 0.25, None, 0,  sp.Symbol("L3"), None, None], 
        [0.50, 0.25, 1, sp.Symbol("R6"),  sp.Symbol("L6") , None, None],
        [0.75, 0.25, 2, sp.Symbol("R9"),  sp.Symbol("L9") , None, None],
        [1.00, 0.25, 3, sp.Symbol("R12"), sp.Symbol("L12"), None, None],
    ]
)

df_swap_market["Discount"] = [
    sp.exp(-x["Spot rate"]*x["Maturity"]) for i, x in df_swap_market.iterrows()
    ]

# Assuming the floating leg cashflow is par yield bond (PV=100, Principal=100)
df_swap_market["Par yield PV"] = [
    sp.Eq(100,(x["Swap rate"] * x["Frequency"] * df_swap_market["Discount"].loc[:i]).sum() + 100 * x["Discount"]) for i, x in df_swap_market.iterrows()
    ]

print("System of equations")
for eqn in df_swap_market["Par yield PV"].iloc[1:].tolist():
    display(eqn)

display(df_swap_market[["Maturity", "Swap rate", "Spot rate", "Discount", "Forward rate"]])
    

System of equations


Eq(100, 0.25 + 100.25*exp(-0.5*R6))

Eq(100, 0.5 + 100.5*exp(-0.75*R9) + 0.5*exp(-0.5*R6))

Eq(100, 0.75 + 0.75*exp(-0.75*R9) + 0.75*exp(-0.5*R6) + 100.75*exp(-1.0*R12))

,Maturity,Swap rate,Spot rate,Discount,Forward rate
0,0.25,NaN,0,1,L3
1,0.50,1.0,R6,exp(-0.5*R6),L6
2,0.75,2.0,R9,exp(-0.75*R9),L9
3,1.00,3.0,R12,exp(-1.0*R12),L12


In [40]:
print("Bootstraping")
df_swap_market.loc[0, "Spot rate soln"] = 0
df_swap_market.loc[0, "Discount soln"] = 1
df_swap_market.loc[0, "Forward rate soln"] = 0 
for i, x in df_swap_market.iloc[1:].iterrows():
    eqn = x["Par yield PV"]
    # Substitute known solution
    for j, y in df_swap_market.iloc[1:i].iterrows():
        eqn = eqn.subs(y["Spot rate"], y["Spot rate soln"])
    display(eqn)
    soln = sp.nsolve(eqn, x["Spot rate"], 0)
    df_swap_market.loc[i,"Spot rate soln"] = float(soln)

    # Discount factor
    df_swap_market.loc[i,"Discount soln"] = float(x["Discount"].subs(x["Spot rate"], soln))

df_swap_market["Spot rate soln"] = df_swap_market["Spot rate soln"].astype(float)
df_swap_market["Discount soln"] = df_swap_market["Discount soln"].astype(float)

display(df_swap_market[["Maturity", "Swap rate", "Spot rate soln", "Discount soln", "Forward rate"]])

Bootstraping


Eq(100, 0.25 + 100.25*exp(-0.5*R6))

Eq(100, 0.997506234413965 + 100.5*exp(-0.75*R9))

Eq(100, 2.23508393196114 + 100.75*exp(-1.0*R12))

,Maturity,Swap rate,Spot rate soln,Discount soln,Forward rate
0,0.25,NaN,0.000000,1.000000,NaN
1,0.50,1.0,0.010000,0.995012,0.020000
2,0.75,2.0,0.020017,0.985099,0.040051
3,1.00,3.0,0.030076,0.970371,0.060255


### 2. Deriving swap rate from the forward rate

Decomposition of swap contract into FRA contracts with multiple tenor

In [90]:
## Forward rate : Estimated 3M, 6M, 9M forward CP91 rates at t=0
# 3M Forward rate from spot rates 

## Notation
# L6 : Prediction for CD91 at T=3M, as of T=0M. 
# Trade time (T=0M): E[CD91(T=3M)|T=0]
# Settle time (T=3M, fixing in advance) : E[CD91(T=3M)|T=3M] = CD91(T=3M)
# Payment time (T=6M) : CD91(T=3M)

# Forward rates estimation
df_swap_market.loc[0, "Forward rate"] = 0
df_swap_market["Forward rate"] = np.log(df_swap_market["Discount soln"].shift(1)/(df_swap_market["Discount soln"]))
df_swap_market["Forward rate"] /= df_swap_market["Frequency"]

## Verification of forward rate, using FRA pricing
# Forward rates are underlying rates of FRA
# Decomposition into FRA : 3x3, 6x3, 6x3 
# Par FRA : NPV of FRA is 0
df_swap_market["FRA_floating_PV"] = df_swap_market.eval("100 * `Forward rate` * `Frequency` * `Discount soln`").cumsum()
for i, x in df_swap_market.iloc[1:].iterrows():
    df_swap_market.loc[i,"FRA_fixed_PV"] = (x["Swap rate"] * df_swap_market.iloc[:i+1].eval("`Frequency` * `Discount soln`")).sum()

display(df_swap_market[["Maturity","Swap rate","Spot rate soln","Discount soln","Forward rate","FRA_floating_PV", "FRA_fixed_PV"]])
print()
print("Error: Fixed leg PV is a bit higher (0.1bp ~ 1bp). Maybe risk adjustments?")
display(df_swap_market.eval("`FRA_floating_PV` - `FRA_fixed_PV`").rename("NPV(%), fixed payer").to_frame())

,Maturity,Swap rate,Spot rate soln,Discount soln,Forward rate,FRA_floating_PV,FRA_fixed_PV
0,0.25,NaN,0.000000,1.000000,NaN,NaN,NaN
1,0.50,1.0,0.010000,0.995012,0.020000,0.497507,0.498753
2,0.75,2.0,0.020017,0.985099,0.040051,1.483856,1.490056
3,1.00,3.0,0.030076,0.970371,0.060255,2.945597,2.962862



Error: Fixed leg PV is a bit higher (0.1bp ~ 1bp). Maybe risk adjustments?


,"NPV(%), fixed payer"
0,NaN
1,-0.001246
2,-0.006200
3,-0.017265


## Hypothetical case study

Swap contract, Fixing in advance

### Cashflow schedule of fixed rate payer

| | Maturity |	Swap rate	|Spot rate	|Discount	|Forward rate	|FRA_floating_PV|	FRA_fixed_PV |
|--|--|--|--|--|--|--|--|
| 3	| 1.00	|3.0	|0.03	|0.97	|0.06|	2.95|	2.96|


Underlying rates, Forward (Esitmated at t=0)

| TxM | E[CD91(t=T)\|t=0] |
|--|--|
| 0x3 | 0	 |
| 3x3 |	0.020000 |
| 6x3 |	0.040051 |
| 9x3 |	0.060255 |


* 2024-12
  * Transaction : Agreed to pay X = 3% p.a with frequency 3M, receive LIBOR 3M with frequency 3M. Maturity 1y.
    * (common practice for the fixed leg payment frequency is 6M)
    * Cashflows are estimated from the forward rates. PV of each leg are discounted using zero rates. For par contract, NPV ~ 0 
    * Fixed payer think : "Based on forward rates, I pay 2.96 in PV, and receive 2.95 in PV. I agree to do that" 
  * Settlement : L3 = 0 (in this case)

* 2025-03
  * Settlement : L6 
    * L6 was estimated at the transaction
      * E[L6\|T=0] = 2.00 @ 2024-12
    * Let's say L6 is fixed
      * Let's assume that an event (e.g. high inflation) caused the rate to be higher than the expected 
      * E[L6\|T=3] = 2.50 @ 2025-03
  * Payment : X/4 - (L3)/4 = 0.75

* 2025-06
  * Settlement
    * E[L9\|T=0] = 4.01
    * E[L9\|T=6] = 4.50
  * Payment : X/4 - E[L6\|T=3]/4 = 0.75 - 0.625 = 0.125

* 2025-09
  * Settlement
    * E[L12\|T=0] = 6.03
    * E[L12\|T=6] = 6.00
  * Payment : X/4 - E[L9\|T=6]/4 = 0.75 - 1.125 = -0.375

* 2025-09
  * Payment : X/4 - E[L12\|T=9]/4 = 0.75 - 1.5 = -0.75
  * Current balance : 
    * Realized rates (assumed) : E[L3\|T=3], E[L6\|T=6], E[L9\|T=9], E[L12\|T=12] = [0.0 , 2.60, 4.60, 5.90]
    * Discounts : [1.026, 1.026, 1.075, 1.140 ]
    * Floating(received) cashflow PV @ 2025-12 : 0 * 1.026 + 0.625 * 1.026 + 1.125 * 1.075  + 1.5 * 1.140 = 3.56
    * Fixed(paid) cashflow PV @ 2025-12 :  0.75 * 1.026 + 0.75 * 1.026 + 0.75 * 1.075  + 0.75 * 1.140 = 3.20
    * Realized interest rate is higher than the expected, so the fixed rate payer is winner in this case.